# SIC8 — 03 Historical Silver Processing · v1.0

**Goal:** Rebuild the verified 24,386,900-row historical snapshot into governed Delta Silver tables. The original five CSV files remain unchanged.

**Read before running:** This version is configured for a **first publication**. It validates the full source, then creates seven managed Delta tables. If any destination table already exists, it stops rather than overwriting it. Run the notebook **from the first cell, once**, and export the results for review. No streaming data is processed here.

**Source semantics:** `Online → ECOMMERCE` (inferred), `Swipe → POS` and `Chip → ATM` (project assumptions). Exact transaction GPS, UTC timezone, and banking-native IDs are not present in the historical source; none are fabricated as source facts.


## 01 — Configuration
Centralize paths and publication settings. `ENABLE_WRITE=True` runs the full quality gates **before** creating new managed tables. The verified source size and ID checksum are used as snapshot-drift guards.

In [0]:
from pyspark.sql import functions as F, types as T, Window
from pyspark.sql.functions import broadcast
import time
import uuid

CATALOG = "workspace"
INPUT_SCHEMA = "sic8_sources"
SILVER_SCHEMA = f"{CATALOG}.sic8_silver"  # Proposed managed schema
RAW = f"/Volumes/{CATALOG}/{INPUT_SCHEMA}/historical_raw"

FILES = {
    "transactions": f"{RAW}/credit_card_transactions-ibm_v2.csv",
    "users": f"{RAW}/sd254_users.csv",
    "cards": f"{RAW}/sd254_cards.csv",
    "mcc": f"{RAW}/merchant_category_codes.csv",
    "cities": f"{RAW}/city_reference.csv",
}

MODE = "FULL"            # FULL for production-sized validation; SAMPLE for code exploration only
SAMPLE_ROWS = 5000
ENABLE_WRITE = True      # First publication only; existing tables are never overwritten
SOURCE_VERSION = "IBM_V2"  # Project label; not an S3 object version or checksum
MAPPING_VERSION = "silver_historical_1.0"

EXPECTED_SOURCE_ROWS = 24386900
EXPECTED_USERS_ROWS = 2000
EXPECTED_TX_ID_CHECKSUM = "1845018497720548795465"  # Previous verified full-run checksum
RUN_TOKEN = str(uuid.uuid4())
VALIDATED_RUN_TOKEN = None
FINAL_GATES_PASSED = False

if ENABLE_WRITE and MODE != "FULL":
    raise ValueError("Publication requires FULL mode")
if MODE not in ("SAMPLE", "FULL"):
    raise ValueError("MODE must be SAMPLE or FULL")
# Fail before heavy processing if any destination already exists.
if ENABLE_WRITE:
    planned_names = [
        "historical_transactions", "historical_ground_truth", "historical_quarantine",
        "historical_users", "historical_cards", "historical_mcc", "historical_cities"
    ]
    existing_at_start = [
        name for name in planned_names
        if spark.catalog.tableExists(f"{SILVER_SCHEMA}.{name}")
    ]
    if existing_at_start:
        raise RuntimeError(f"Destination tables already exist: {existing_at_start}")
print("Mode:", MODE, "| Target:", SILVER_SCHEMA, "| Write enabled:", ENABLE_WRITE)

Mode: FULL | Target: workspace.sic8_silver | Write enabled: True


## 02 — Read the five original CSV sources
Read each file with its own schema as strings. No data is modified in S3.

In [0]:
def read_csv(path):
    return (spark.read
        .option("header", "true")
        .option("inferSchema", "false")
        .option("enforceSchema", "false")
        .csv(path))

raw = {name: read_csv(path) for name, path in FILES.items()}
# Normalize a possible UTF-8 BOM in reference headers (e.g. mcc).
raw = {name: frame.toDF(*[col.lstrip("\ufeff") for col in frame.columns])
       for name, frame in raw.items()}

TX_COLUMNS = [
    "User", "Card", "Year", "Month", "Day", "Time", "Amount",
    "Use Chip", "Merchant Name", "Merchant City", "Merchant State",
    "Zip", "MCC", "Errors?", "Is Fraud?"
]

if raw["transactions"].columns != TX_COLUMNS:
    raise ValueError("Unexpected transaction columns; verify CSV version")

for name, frame in raw.items():
    print(f"{name}: {len(frame.columns)} columns")

source_transactions = (raw["transactions"].limit(SAMPLE_ROWS)
    if MODE == "SAMPLE" else raw["transactions"])

transactions: 15 columns
users: 18 columns
cards: 13 columns
mcc: 5 columns
cities: 7 columns


## 03 — Check source contracts
Stop early if required historical/reference columns are absent.

In [0]:
required = {
    "users": ["Person", "City", "State"],
    "cards": ["User", "CARD INDEX", "Card Brand", "Card Type", "Has Chip"],
    "mcc": ["mcc", "description", "category"],
    "cities": ["city_name", "state_code", "country_code", "latitude", "longitude"],
}
for name, fields in required.items():
    missing = [field for field in fields if field not in raw[name].columns]
    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")

print("Reference schemas: OK")
print("Users join needs verified file-order mapping; see Section 04.")

Reference schemas: OK
Users join needs verified file-order mapping; see Section 04.


## 04 — Reconstruct Users source indices
The historical `User` field is the zero-based position of a Users CSV record. The file is small enough to read as bytes; the **2.35 GB transaction CSV must not** be treated this way. Complete content equality is tested in the final gates.

In [0]:
# Read the original Users file while preserving row order
users_binary = spark.read.format("binaryFile").load(FILES["users"])

users_lines = (
    users_binary
    .select(
        F.posexplode(
            F.split(
                F.decode(F.col("content"), "UTF-8"),
                r"\r?\n"
            )
        ).alias("line_no", "csv_line")
    )
    .filter(
        (F.col("line_no") > 0) &
        (F.trim(F.col("csv_line")) != "")
    )
)

# Convert StructType into a DDL schema string
users_ddl = raw["users"].schema.toDDL()

users_indexed = (
    users_lines
    .select(
        (F.col("line_no") - 1)
            .cast("string")
            .alias("_user_key"),

        F.from_csv(
            F.col("csv_line"),
            users_ddl
        ).alias("user_record")
    )
    .select("_user_key", "user_record.*")
)

# Validate reconstructed records
raw_user_count = raw["users"].count()
indexed_user_count = users_indexed.count()

if raw_user_count != indexed_user_count:
    raise ValueError("User row count mismatch")

if users_indexed.filter(F.col("Person").isNull()).limit(1).count():
    raise ValueError("Users CSV parsing failed")

if users_indexed.filter(F.col("_user_key").isNull()).limit(1).count():
    raise ValueError("Missing user indices")

users_ref = users_indexed.select(
    "_user_key",
    F.col("City").alias("customer_city"),
    F.col("State").alias("customer_region"),
    F.lit(True).alias("user_found")
)

print("Original users:", raw_user_count)
print("Indexed users:", indexed_user_count)

display(
    users_indexed.select(
        "_user_key", "City", "State"
    ).limit(5)
)

Original users: 2000
Indexed users: 2000


_user_key,City,State
0,La Verne,CA
1,Little Neck,NY
2,West Covina,CA
3,New York,NY
4,San Francisco,CA


## 05 — Build safe reference lookups
Reject non-unique Card/MCC keys. Join only unambiguous US city+state centroids, never treat them as transaction GPS.

In [0]:
ref_cards = raw["cards"].select(
    F.trim(F.col("User")).alias("_user_key"),
    F.trim(F.col("CARD INDEX")).alias("_card_key"),
    F.col("Card Brand").alias("card_brand"),
    F.col("Card Type").alias("card_type"),
    F.col("Has Chip").alias("has_chip_raw"),
    F.lit(True).alias("card_found")
)

ref_mcc = raw["mcc"].select(
    F.lpad(F.trim(F.col("mcc")), 4, "0").alias("mcc_key"),
    F.col("description").alias("mcc_description"),
    F.col("category").alias("merchant_category"),
    F.lit(True).alias("mcc_found")
)

ref_city_all = (raw["cities"]
    .filter(F.upper(F.trim(F.col("country_code"))) == "US")
    .select(
        F.upper(F.trim(F.col("city_name"))).alias("_city_key"),
        F.upper(F.trim(F.col("state_code"))).alias("_region_key"),
        F.expr("try_cast(latitude AS DOUBLE)").alias("merchant_city_centroid_lat"),
        F.expr("try_cast(longitude AS DOUBLE)").alias("merchant_city_centroid_lon")
    ))

# Only unambiguous city+state keys are allowed in a transaction join.
city_counts = ref_city_all.groupBy("_city_key", "_region_key").count()
unique_city_keys = city_counts.filter(F.col("count") == 1).drop("count")
ref_cities = ref_city_all.join(unique_city_keys, ["_city_key", "_region_key"], "inner")

checks = {
    "duplicate_card_keys": ref_cards.groupBy("_user_key", "_card_key").count()
        .filter(F.col("count") > 1).count(),
    "duplicate_mcc_keys": ref_mcc.groupBy("mcc_key").count()
        .filter(F.col("count") > 1).count(),
    "ambiguous_us_city_keys": city_counts.filter(F.col("count") > 1).count(),
}
if checks["duplicate_card_keys"] or checks["duplicate_mcc_keys"]:
    raise ValueError(f"Non-unique card/MCC reference keys: {checks}")
if users_ref.groupBy("_user_key").count().filter(F.col("count") > 1).count():
    raise ValueError("Duplicate reconstructed user indices")

print("Reference checks:", checks)
print("Excluded ambiguous US city keys from enrichment.")

Reference checks: {'duplicate_card_keys': 0, 'duplicate_mcc_keys': 0, 'ambiguous_us_city_keys': 0}
Excluded ambiguous US city keys from enrichment.


## 06 — Channel contract
Keep `payment_method` (source-derived) separate from `channel` (inferred/assumed). The `channel_origin` field documents that distinction.

In [0]:
def add_payment_mapping(df):
    method = (F.when(F.trim(F.col("Use Chip")) == "Swipe Transaction", "SWIPE")
        .when(F.trim(F.col("Use Chip")) == "Chip Transaction", "CHIP")
        .when(F.trim(F.col("Use Chip")) == "Online Transaction", "ONLINE")
        .otherwise("UNKNOWN"))
    return (df
        .withColumn("payment_method", method)
        .withColumn("channel", F.when(F.col("payment_method") == "ONLINE", "ECOMMERCE")
            .when(F.col("payment_method") == "SWIPE", "POS")
            .when(F.col("payment_method") == "CHIP", "ATM")
            .otherwise("UNKNOWN"))
        .withColumn("channel_origin", F.when(F.col("payment_method") == "ONLINE", "INFERRED_FROM_METHOD")
            .when(F.col("payment_method").isin("SWIPE", "CHIP"), "PROJECT_ASSUMPTION")
            .otherwise("UNKNOWN"))
        .withColumn("card_present", F.when(F.col("payment_method").isin("SWIPE", "CHIP"), F.lit(True))
            .when(F.col("payment_method") == "ONLINE", F.lit(False))
            .otherwise(F.lit(None).cast("boolean"))))

fixture = spark.createDataFrame([
    ("Swipe Transaction",), ("Chip Transaction",),
    ("Online Transaction",), ("other",)
], ["Use Chip"])
actual = [(x["payment_method"], x["channel"]) for x in
          add_payment_mapping(fixture).select("payment_method", "channel").collect()]
assert actual == [("SWIPE", "POS"), ("CHIP", "ATM"),
                  ("ONLINE", "ECOMMERCE"), ("UNKNOWN", "UNKNOWN")]
print("Channel rules passed 4 fixture checks.")

Channel rules passed 4 fixture checks.


## 07 — Historical row identity
Create a content fingerprint **without the fraud label**; assign an occurrence number for byte-equivalent transaction fields. With identical non-label rows and identical labels, the **set of IDs** is repeatable across runs, but no original physical-row ID can be recovered. The final gate rejects mixed-label identical-feature groups.

In [0]:
non_label_fields = [F.col(name).alias(f"field_{i:02d}")
                    for i, name in enumerate(TX_COLUMNS[:-1], start=1)]

source_hashed = source_transactions.withColumn(
    "source_row_hash", F.sha2(F.to_json(F.struct(*non_label_fields)), 256))

# Equal content hashes have no original source row ID to order by.
# All copies in each hash group are indistinguishable; label consistency is gated later.
occurrence_window = (Window.partitionBy("source_row_hash")
    .orderBy(F.lit(1)))

source_identified = (source_hashed
    .withColumn("_duplicate_occurrence", F.row_number().over(occurrence_window))
    .withColumn("source_record_id", F.sha2(F.concat(
        F.lit(SOURCE_VERSION + "|"), F.col("source_row_hash"), F.lit("|"),
        F.col("_duplicate_occurrence").cast("string")), 256))
    .withColumn("transaction_id", F.concat(F.lit("HIST_TX_"), F.col("source_record_id")))
    .withColumn("event_id", F.concat(F.lit("HIST_EVENT_"), F.col("source_record_id"))))

print("Historical keys are project-derived, not source IDs.")

Historical keys are project-derived, not source IDs.


## 08 — Type conversion and canonical fields
Keep financial values as decimal, timestamps as local `TIMESTAMP_NTZ`, and source/channel provenance explicit.

In [0]:
typed = (source_identified
    .withColumn("_user_key", F.trim(F.col("User")))
    .withColumn("_card_key", F.trim(F.col("Card")))
    .withColumn("source_merchant_id", F.trim(F.col("Merchant Name")))
    .withColumn("customer_id", F.when(
        F.col("_user_key").rlike(r"^[0-9]+$"),
        F.concat(F.lit("IBM_USER_"), F.col("_user_key"))))
    .withColumn("card_id", F.when(
        F.col("_user_key").rlike(r"^[0-9]+$") & F.col("_card_key").rlike(r"^[0-9]+$"),
        F.concat(F.lit("IBM_CARD_"), F.col("_user_key"), F.lit("_"), F.col("_card_key"))))
    .withColumn("amount", F.expr(
        "try_cast(regexp_replace(trim(`Amount`), '[$,]', '') AS DECIMAL(18,2))"))
    .withColumn("currency", F.lit("USD"))
    .withColumn("_local_timestamp_text", F.expr("""
        concat(lpad(trim(`Year`),4,'0'),'-',lpad(trim(`Month`),2,'0'),'-',
               lpad(trim(`Day`),2,'0'),' ',trim(`Time`))
    """))
    .withColumn("source_event_time_local", F.expr(
        "try_cast(_local_timestamp_text AS TIMESTAMP_NTZ)"))
    .withColumn("event_time_timezone_status", F.lit("UNKNOWN"))
    .withColumn("mcc_key", F.lpad(F.trim(F.col("MCC")), 4, "0"))
    .withColumn("merchant_city", F.col("Merchant City"))
    .withColumn("merchant_region_raw", F.col("Merchant State"))
    .withColumn("merchant_postal_code", F.regexp_replace(
        F.trim(F.col("Zip")), r"\.0$", ""))
    .withColumn("_city_key", F.upper(F.trim(F.col("Merchant City"))))
    .withColumn("_region_key", F.upper(F.trim(F.col("Merchant State"))))
    .withColumn("source_system", F.lit("IBM_KAGGLE_HISTORICAL"))
    .withColumn("source_dataset_version", F.lit(SOURCE_VERSION))
    .withColumn("mapping_version", F.lit(MAPPING_VERSION))
    .withColumn("ingestion_mode", F.lit("BATCH"))
    .withColumn("errors_raw", F.col("Errors?"))
    .withColumn("outcome_status", F.lit("UNKNOWN"))
    .withColumn("transaction_type", F.lit("UNKNOWN"))
    .withColumn("source_file", F.lit(FILES["transactions"])))

mapped = add_payment_mapping(typed)
print("Amount, local time, identity and project channels mapped.")

Amount, local time, identity and project channels mapped.


## 09 — Reference enrichment
Left joins preserve every transaction, even when optional reference attributes are absent.

In [0]:
enriched = (mapped
    .join(broadcast(users_ref), ["_user_key"], "left")
    .join(broadcast(ref_cards), ["_user_key", "_card_key"], "left")
    .join(broadcast(ref_mcc), ["mcc_key"], "left")
    .join(broadcast(ref_cities), ["_city_key", "_region_key"], "left")
    .withColumn("latitude", F.lit(None).cast("double"))
    .withColumn("longitude", F.lit(None).cast("double"))
    .withColumn("location_type", F.lit("TRANSACTION_LOCATION_NOT_RECORDED"))
    .withColumn("merchant_location_precision", F.when(
        F.col("merchant_city_centroid_lat").isNotNull() &
        F.col("merchant_city_centroid_lon").isNotNull(), "CITY_CENTROID"
    ).otherwise("UNKNOWN"))
    .withColumn("geo_eligible_for_exact_velocity", F.lit(False))
    .withColumn("atm_mcc_evidence", F.col("mcc_key") == "6011"))

print("Reference enrichments prepared; exact transaction GPS remain NULL.")

Reference enrichments prepared; exact transaction GPS remain NULL.


## 10 — Business data-quality rules
Only invalid required fields send rows to Quarantine. Negative amounts and missing optional location information remain valid.

In [0]:
checked = (enriched
    .withColumn("bad_identity", F.col("customer_id").isNull() | F.col("card_id").isNull())
    .withColumn("bad_amount", F.col("amount").isNull())
    .withColumn("bad_local_time", F.col("source_event_time_local").isNull())
    .withColumn("bad_payment_method", F.col("payment_method") == "UNKNOWN")
    .withColumn("bad_fraud_label", ~F.trim(F.coalesce(
        F.col("Is Fraud?"), F.lit(""))).isin("Yes", "No"))
    .withColumn("quality_status", F.when(
        F.col("bad_identity") | F.col("bad_amount") | F.col("bad_local_time") |
        F.col("bad_payment_method") | F.col("bad_fraud_label"), "REVIEW"
    ).otherwise("PASS"))
    .withColumn("quality_reason", F.concat_ws(" | ",
        F.when(F.col("bad_identity"), "BAD_IDENTITY"),
        F.when(F.col("bad_amount"), "INVALID_AMOUNT"),
        F.when(F.col("bad_local_time"), "INVALID_LOCAL_TIME"),
        F.when(F.col("bad_payment_method"), "UNKNOWN_METHOD"),
        F.when(F.col("bad_fraud_label"), "INVALID_SOURCE_LABEL")))
    .withColumn("ground_truth_label", F.when(F.col("Is Fraud?") == "Yes", F.lit(True))
        .when(F.col("Is Fraud?") == "No", F.lit(False))
        .otherwise(F.lit(None).cast("boolean"))))

accepted = checked.filter(F.col("quality_status") == "PASS")
rejected = checked.filter(F.col("quality_status") == "REVIEW")
print("Quality rules created. Missing card / MCC / city lookups stay visible as NULL.")

Quality rules created. Missing card / MCC / city lookups stay visible as NULL.


## 11 — Output contracts and reference dimensions
Fraud labels remain in a separate sidecar. Additional **Silver Users, Cards, MCC, and Cities** tables support downstream dimension modeling and ML features; no card numbers, CVVs, user names, or postal addresses are exposed.

In [0]:
TRANSACTION_FIELDS = [
    "source_record_id", "transaction_id", "event_id", "source_row_hash",
    "source_system", "source_dataset_version", "mapping_version", "ingestion_mode", "source_file",
    "customer_id", "card_id", "card_brand", "card_type", "has_chip_raw", "source_merchant_id",
    "source_event_time_local", "event_time_timezone_status",
    "amount", "currency", "payment_method", "channel", "channel_origin", "card_present", "transaction_type",
    "mcc_key", "mcc_description", "merchant_category", "merchant_city", "merchant_region_raw",
    "merchant_postal_code", "customer_city", "customer_region",
    "latitude", "longitude", "merchant_city_centroid_lat", "merchant_city_centroid_lon",
    "merchant_location_precision", "location_type", "geo_eligible_for_exact_velocity",
    "outcome_status", "errors_raw", "atm_mcc_evidence"
]

silver_transactions = accepted.select(*TRANSACTION_FIELDS)
silver_ground_truth = accepted.select(
    "source_record_id", "transaction_id", "event_id", "ground_truth_label",
    F.lit("IBM_SOURCE_LABEL").alias("label_source"))
silver_quarantine = rejected.select(
    "source_record_id", "quality_reason", "source_system", "source_dataset_version",
    *[F.col(field).alias(f"raw_{i:02d}") for i, field in enumerate(TX_COLUMNS, 1)]
)

assert "ground_truth_label" not in silver_transactions.columns
assert "Is Fraud?" not in silver_transactions.columns
assert "Card Number" not in silver_transactions.columns
assert "CVV" not in silver_transactions.columns
print("Contracts:", len(silver_transactions.columns), "transaction fields;",
      len(silver_ground_truth.columns), "ground-truth fields")

Contracts: 42 transaction fields; 5 ground-truth fields


In [0]:
# Publish reference entities separately instead of hiding them in joins.
silver_users = users_indexed.select(
    F.concat(F.lit("IBM_USER_"), F.col("_user_key")).alias("customer_id"),
    F.col("_user_key").cast("long").alias("source_user_index"),
    F.col("City").alias("customer_city"),
    F.col("State").alias("customer_region"),
    F.lit("IBM_KAGGLE_HISTORICAL").alias("source_system")
)

# Include optional customer ML attributes only if they exist in this source version.
user_optional = {
    "FICO Score": ("fico_score", "INT"),
    "Current Age": ("current_age", "INT"),
    "Num Credit Cards": ("num_credit_cards", "INT"),
    "Yearly Income": ("yearly_income", "DECIMAL(18,2)"),
    "Total Debt": ("total_debt", "DECIMAL(18,2)"),
    "Latitude": ("customer_home_latitude_reference", "DOUBLE"),
    "Longitude": ("customer_home_longitude_reference", "DOUBLE")
}
for input_col, (output_col, target_type) in user_optional.items():
    if input_col in users_indexed.columns:
        if target_type.startswith("DECIMAL"):
            expression = f"try_cast(regexp_replace(trim(`{input_col}`), '[$,]', '') AS {target_type})"
        else:
            expression = f"try_cast(`{input_col}` AS {target_type})"
        # Append the typed optional attribute to the Users source projection.
        # The source column was omitted from the initial minimal projection.
        optional_feature = users_indexed.select(
            F.col("_user_key").cast("long").alias("source_user_index"),
            F.expr(expression).alias(output_col)
        )
        silver_users = silver_users.join(broadcast(optional_feature), "source_user_index", "left")

In [0]:
silver_cards = ref_cards.select(
    F.concat(F.lit("IBM_CARD_"), F.col("_user_key"), F.lit("_"), F.col("_card_key")).alias("card_id"),
    F.concat(F.lit("IBM_USER_"), F.col("_user_key")).alias("customer_id"),
    F.col("_card_key").alias("source_card_index"),
    "card_brand", "card_type", "has_chip_raw",
    F.lit("IBM_KAGGLE_HISTORICAL").alias("source_system")
)
if "Credit Limit" in raw["cards"].columns:
    card_limit = raw["cards"].select(
        F.concat(F.lit("IBM_CARD_"), F.trim(F.col("User")), F.lit("_"), F.trim(F.col("CARD INDEX"))).alias("card_id"),
        F.expr("try_cast(regexp_replace(trim(`Credit Limit`), '[$,]', '') AS DECIMAL(18,2))").alias("credit_limit")
    )
    silver_cards = silver_cards.join(broadcast(card_limit), "card_id", "left")

silver_mcc = ref_mcc.select(
    F.col("mcc_key").alias("mcc"), "mcc_description", "merchant_category"
)

silver_cities = ref_cities.select(
    F.col("_city_key").alias("city_name_normalized"),
    F.col("_region_key").alias("region_code"),
    "merchant_city_centroid_lat", "merchant_city_centroid_lon"
)
print("Prepared Silver dimensions: users, cards, MCC, city centroids")

Prepared Silver dimensions: users, cards, MCC, city centroids


## 12 — Full reconciliation
Count original and joined rows. Confirm all transactions remain in PASS/REVIEW and examine the project-defined ATM split.

In [0]:
start = time.perf_counter()
source_count = source_transactions.count()
summary = checked.agg(
    F.count("*").alias("rows_after_joins"),
    F.sum((F.col("quality_status") == "PASS").cast("long")).alias("accepted"),
    F.sum((F.col("quality_status") == "REVIEW").cast("long")).alias("review"),
    F.sum((F.col("amount") < 0).cast("long")).alias("negative_amounts"),
    F.sum((F.col("ground_truth_label") == True).cast("long")).alias("fraud_yes"),
    F.sum(F.col("card_found").isNull().cast("long")).alias("missing_card_reference"),
    F.sum(F.col("user_found").isNull().cast("long")).alias("missing_user_reference"),
    F.sum(F.col("mcc_found").isNull().cast("long")).alias("missing_mcc_reference"),
    F.sum((F.col("channel") == "ATM").cast("long")).alias("project_assigned_atm"),
    F.sum(((F.col("channel") == "ATM") &
            (F.col("mcc_key") != "6011")).cast("long")).alias("atm_without_mcc_6011")
).first().asDict()

if summary["rows_after_joins"] != source_count:
    raise AssertionError("Join changed the number of source transactions")
if summary["accepted"] + summary["review"] != source_count:
    raise AssertionError("Quality split lost rows")

print("Validation metrics:")
for k, v in {"source_rows": source_count, **summary}.items():
    print(f"  {k}: {v}")
print("Validation seconds:", round(time.perf_counter() - start, 2))
print("WARNING: project-assigned ATM rows are not proof of real ATM transactions.")

Validation metrics:
  source_rows: 24386900
  rows_after_joins: 24386900
  accepted: 24386900
  review: 0
  negative_amounts: 1244683
  fraud_yes: 29757
  missing_card_reference: 0
  missing_user_reference: 0
  missing_mcc_reference: 0
  project_assigned_atm: 6287598
  atm_without_mcc_6011: 6287598
Validation seconds: 32.19


## 13 — Safe preview
Preview normalized business fields without labels or sensitive reference fields.

In [0]:
display(silver_transactions.select(
    "transaction_id", "customer_id", "card_id", "amount", "currency",
    "payment_method", "channel", "channel_origin", "merchant_category",
    "merchant_location_precision", "geo_eligible_for_exact_velocity"
).limit(10))
print("Ground truth remains separate:", silver_ground_truth.columns)

transaction_id,customer_id,card_id,amount,currency,payment_method,channel,channel_origin,merchant_category,merchant_location_precision,geo_eligible_for_exact_velocity
HIST_TX_44c498312be7d14d5c017f7c37d08caf94871a3d20c03a6da8295ef96a3ba185,IBM_USER_896,IBM_CARD_896_4,14.76,USD,SWIPE,POS,PROJECT_ASSUMPTION,Miscellaneous Stores,CITY_CENTROID,false
HIST_TX_89722fd412f099c5f35d3b4c3910c7a24fbfc70e9209ff4121c78631e06d5d72,IBM_USER_801,IBM_CARD_801_3,24.91,USD,ONLINE,ECOMMERCE,INFERRED_FROM_METHOD,Transportation Services,UNKNOWN,false
HIST_TX_066484cf6fdea60a569a7f8067fb409b75681e62681a289489cff89b616745d1,IBM_USER_531,IBM_CARD_531_0,33.31,USD,SWIPE,POS,PROJECT_ASSUMPTION,Miscellaneous Stores,CITY_CENTROID,false
HIST_TX_3c28f05dd876e7631fb7e569aaccbd18288bacaf57836d61ad35790bcf68c80e,IBM_USER_134,IBM_CARD_134_4,62.49,USD,ONLINE,ECOMMERCE,INFERRED_FROM_METHOD,Business Services,UNKNOWN,false
HIST_TX_66132cc16e463295b822d7696731167f82b395c0c93dfb1f26206afd6e4b9d11,IBM_USER_387,IBM_CARD_387_0,1.79,USD,CHIP,ATM,PROJECT_ASSUMPTION,Retail Outlet Services,CITY_CENTROID,false
HIST_TX_a19a18e134376c79d7e7679e8c66e7ffd251081c55d5c66f5f33067bd7d63f44,IBM_USER_50,IBM_CARD_50_0,80.00,USD,SWIPE,POS,PROJECT_ASSUMPTION,Utility Services,UNKNOWN,false
HIST_TX_dec3d89b8edd09b33064ba5556d6981e6e053e3556b57cfacd7bc4aaa7e66dfe,IBM_USER_1710,IBM_CARD_1710_0,174.83,USD,SWIPE,POS,PROJECT_ASSUMPTION,Utility Services,UNKNOWN,false
HIST_TX_2cd2ba923bb712a7dc18d830d7d6e2ccf630a22954032d0e385d131465c81152,IBM_USER_956,IBM_CARD_956_0,58.00,USD,SWIPE,POS,PROJECT_ASSUMPTION,Retail Outlet Services,CITY_CENTROID,false
HIST_TX_6c9ccfe9b7e7047e7715b0098bb9ad69b60d35fd74dfbc165239f391b236d57f,IBM_USER_1940,IBM_CARD_1940_0,43.20,USD,ONLINE,ECOMMERCE,INFERRED_FROM_METHOD,Transportation Services,UNKNOWN,false
HIST_TX_8c0cadcbb62157c2e5161949b8e896933ecaf00969549c4a6334cbd3406ee9a0,IBM_USER_1644,IBM_CARD_1644_4,49.13,USD,SWIPE,POS,PROJECT_ASSUMPTION,Business Services,CITY_CENTROID,false


Ground truth remains separate: ['source_record_id', 'transaction_id', 'event_id', 'ground_truth_label', 'label_source']


## 14 — Final quality gate (single authoritative check)
Reconstruct Users exactly, detect conflicting labels for otherwise identical records, verify all transaction IDs, and check dimensional key uniqueness. The checksum must match the previously verified historical snapshot.

In [0]:
audit_started = time.perf_counter()
FINAL_GATES_PASSED = False
VALIDATED_RUN_TOKEN = None

if MODE != "FULL":
    raise RuntimeError("The publication gate requires the full snapshot")
if source_count != EXPECTED_SOURCE_ROWS:
    raise AssertionError(f"Source changed: {source_count} != {EXPECTED_SOURCE_ROWS}")

cols = raw["users"].columns
missing_user_records = raw["users"].select(*cols).exceptAll(users_indexed.select(*cols)).count()
extra_user_records = users_indexed.select(*cols).exceptAll(raw["users"].select(*cols)).count()
if missing_user_records or extra_user_records or indexed_user_count != EXPECTED_USERS_ROWS:
    raise AssertionError("Users reconstruction does not match source content and expected count")

conflicting_label_groups = (source_hashed.groupBy("source_row_hash")
    .agg(F.countDistinct(F.coalesce(F.col("Is Fraud?"), F.lit("<NULL>"))).alias("label_variants"))
    .filter(F.col("label_variants") > 1).count())
if conflicting_label_groups:
    raise AssertionError("Indistinguishable non-label transaction rows contain conflicting labels")

identity_stats = checked.agg(
    F.count("*").alias("rows"),
    F.count("transaction_id").alias("non_null_ids"),
    F.countDistinct("transaction_id").alias("unique_ids"),
    F.sum(F.xxhash64("transaction_id").cast("decimal(38,0)")).alias("id_checksum")
).first().asDict()

if identity_stats["rows"] != source_count or identity_stats["non_null_ids"] != source_count:
    raise AssertionError("Transaction ID count mismatch")
if identity_stats["unique_ids"] != source_count:
    raise AssertionError("Transaction ID duplicates")
if str(identity_stats["id_checksum"]) != EXPECTED_TX_ID_CHECKSUM:
    raise AssertionError("The historical transaction ID set changed since the verified run")
if summary["rows_after_joins"] != source_count or summary["accepted"] + summary["review"] != source_count:
    raise AssertionError("Row count changed during enrichment or validation")
if summary["accepted"] != EXPECTED_SOURCE_ROWS or summary["review"] != 0:
    raise AssertionError("Unexpected historical review rows for this verified snapshot")
if summary["fraud_yes"] != 29757:
    raise AssertionError("Historical fraud-label total has drifted")

if "ground_truth_label" in silver_transactions.columns or "Is Fraud?" in silver_transactions.columns:
    raise AssertionError("Ground truth leaked into transaction features")
if "ground_truth_label" not in silver_ground_truth.columns:
    raise AssertionError("Ground truth is missing from its sidecar")

# Uniqueness of reference keys prevents future dimension joins from amplifying rows.
for name, frame, key in [
    ("users", silver_users, ["customer_id"]),
    ("cards", silver_cards, ["card_id"]),
    ("mcc", silver_mcc, ["mcc"]),
    ("cities", silver_cities, ["city_name_normalized", "region_code"])
]:
    if frame.groupBy(*key).count().filter(F.col("count") > 1).limit(1).count():
        raise AssertionError(f"Duplicate Silver {name} dimension key")

FINAL_GATES_PASSED = True
VALIDATED_RUN_TOKEN = RUN_TOKEN
print("FINAL QUALITY GATES: PASS")
print("Source rows / accepted / review:", source_count, summary["accepted"], summary["review"])
print("Users:", indexed_user_count, "| Conflicting-label groups:", conflicting_label_groups)
print("Unique IDs:", identity_stats["unique_ids"], "| ID checksum:", identity_stats["id_checksum"])
print("Audit elapsed seconds:", round(time.perf_counter() - audit_started, 2))
print("Note: the IDs represent a reproducible snapshot multiset, not native banking IDs.")

FINAL QUALITY GATES: PASS
Source rows / accepted / review: 24386900 24386900 0
Users: 2000 | Conflicting-label groups: 0
Unique IDs: 24386900 | ID checksum: 1845018497720548795465
Audit elapsed seconds: 41.72
Note: the IDs represent a reproducible snapshot multiset, not native banking IDs.


## 15 — First managed Delta publication
**This cell writes data** when `ENABLE_WRITE=True`. It refuses to overwrite pre-existing tables. Tables are committed individually (not atomically as a group); if a write fails, inspect the published subset before retrying.

In [0]:
OUTPUTS = {
    "historical_transactions": silver_transactions,
    "historical_ground_truth": silver_ground_truth,
    "historical_quarantine": silver_quarantine,
    "historical_users": silver_users,
    "historical_cards": silver_cards,
    "historical_mcc": silver_mcc,
    "historical_cities": silver_cities,
}

if ENABLE_WRITE:
    if not FINAL_GATES_PASSED or VALIDATED_RUN_TOKEN != RUN_TOKEN:
        raise RuntimeError("Run all cells; final quality gate must pass in this exact run")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}")

    already_exists = [name for name in OUTPUTS if spark.catalog.tableExists(f"{SILVER_SCHEMA}.{name}")]
    if already_exists:
        raise RuntimeError(f"Existing Silver tables detected; refusing to overwrite: {already_exists}")

    for short_name, frame in OUTPUTS.items():
        full_name = f"{SILVER_SCHEMA}.{short_name}"
        frame.write.format("delta").mode("error").saveAsTable(full_name)
        print("PUBLISHED:", full_name)
else:
    print("DRY RUN — zero Delta writes")

PUBLISHED: workspace.sic8_silver.historical_transactions
PUBLISHED: workspace.sic8_silver.historical_ground_truth
PUBLISHED: workspace.sic8_silver.historical_quarantine
PUBLISHED: workspace.sic8_silver.historical_users
PUBLISHED: workspace.sic8_silver.historical_cards
PUBLISHED: workspace.sic8_silver.historical_mcc
PUBLISHED: workspace.sic8_silver.historical_cities


## 16 — Verify published tables
Verify row counts, snapshot labels and reference dimensions **from storage**, not merely the in-memory DataFrames. After success, Notebook 04 can use `RUN_MODE='SILVER'` without re-running the 12-row fixture.

In [0]:
if ENABLE_WRITE:
    published_counts = {
        short_name: spark.table(f"{SILVER_SCHEMA}.{short_name}").count()
        for short_name in OUTPUTS
    }
    assert published_counts["historical_transactions"] == summary["accepted"]
    assert published_counts["historical_ground_truth"] == summary["accepted"]
    assert published_counts["historical_quarantine"] == summary["review"]
    assert published_counts["historical_users"] == indexed_user_count
    assert published_counts["historical_cards"] == raw["cards"].count()
    assert published_counts["historical_mcc"] == raw["mcc"].count()
    print("PUBLISHED SILVER VERIFICATION: PASS")
    for name, count in published_counts.items():
        print(f"  {name}: {count:,}")
    print("Next: run Notebook 04 on historical Silver with writes OFF for evaluation.")
else:
    print("No publication requested.")

PUBLISHED SILVER VERIFICATION: PASS
  historical_transactions: 24,386,900
  historical_ground_truth: 24,386,900
  historical_quarantine: 0
  historical_users: 2,000
  historical_cards: 6,146
  historical_mcc: 914
  historical_cities: 1,430
Next: run Notebook 04 on historical Silver with writes OFF for evaluation.
